# ベースモデルを LoRA で指示チューニングする (SFT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/17_llm_training/lora_finetune.ipynb)

NLP クックブック 17-01 ／ LLM の学習と調整

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

指示チューニング前のベースモデルは、質問を渡しても答えて止まるとは限らない。公開の日本語の指示データで、LoRA により少ないメモリで指示チューニング (SFT) を行い、学習前・学習後・公式の指示チューニング版の応答を同じ指示で比べたい。

In [1]:
!pip install -q trl==1.18.0 peft==0.21.2 torchao==0.18.0   # Colab に入っている古い torchao (0.10) のままだと peft の import が失敗する

In [2]:
import json, os, re, time, unicodedata
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

def load_tokenizer(name):
    tokenizer = AutoTokenizer.from_pretrained(name)
    # transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
    # 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
    sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
    tokenizer.backend_tokenizer.model = models.Unigram(
        [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
    return tokenizer

set_seed(0)
base_name = "sbintuitions/sarashina2.2-0.5b"                   # 指示チューニング前のベースモデル（SB Intuitions、MIT License）
inst_name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"     # 同じベースモデルから作られた公式の指示チューニング版 (MIT License)
tokenizer = load_tokenizer(base_name)
print("ベースモデルのチャットテンプレート:", tokenizer.chat_template)

# ベースモデルにはチャットテンプレートがないので、公式の指示チューニング版と同じ形式のものを自分で設定する。
# 形式: <|system|>…</s><|user|>…</s><|assistant|>…</s>（<|user|> などは語彙に追加されたトークン）
tokenizer.chat_template = ("{% for m in messages %}{{ '<|' + m['role'] + '|>' + m['content'] + eos_token }}{% endfor %}"
                           "{% if add_generation_prompt %}{{ '<|assistant|>' }}{% endif %}")
sample = [{"role": "system", "content": "S"}, {"role": "user", "content": "こんにちは"}, {"role": "assistant", "content": "はい。"}]
official = AutoTokenizer.from_pretrained(inst_name)   # 公式のテンプレートと同じ文字列になるかだけを確かめる
assert tokenizer.apply_chat_template(sample, tokenize=False) == official.apply_chat_template(sample, tokenize=False)
print(tokenizer.apply_chat_template(sample[1:2], tokenize=False, add_generation_prompt=True))

BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
DTYPE = torch.bfloat16 if BF16 else torch.float16
model = AutoModelForCausalLM.from_pretrained(base_name, dtype=DTYPE, device_map="auto")
print(f"パラメータ数: {model.num_parameters() / 1e6:.0f}M（埋め込みと出力層を含む）")

ベースモデルのチャットテンプレート: None
<|user|>こんにちは</s><|assistant|>
パラメータ数: 793M（埋め込みと出力層を含む）


In [3]:
from datasets import Dataset, load_dataset

# 指示データ: llm-jp/databricks-dolly-15k-ja (CC BY-SA 3.0)。Databricks の社員が書いた databricks-dolly-15k
# （指示と応答の組、CC BY-SA 3.0）を、LLM-jp が DeepL で日本語に訳したもの。context は Wikipedia から引用した参考文
raw = load_dataset("llm-jp/databricks-dolly-15k-ja", split="train")
print(len(raw), "件", raw.to_pandas()["category"].value_counts().to_dict())

def to_example(r):
    user = r["instruction"] if not r["context"] else f"{r['instruction']}\n\n{r['context']}"
    # trl の「会話形式の prompt-completion」: prompt と completion にメッセージのリストを入れる
    return {"prompt": [{"role": "user", "content": user}], "completion": [{"role": "assistant", "content": r["response"]}]}

# 応答が空でなく、全体が短めのもの（max_length の 512 トークンに収まる長さ）に絞る
rows = [r for r in raw if r["response"].strip() and len(r["instruction"]) + len(r["context"]) + len(r["response"]) <= 600]
rng = np.random.default_rng(0)
rows = [rows[i] for i in rng.permutation(len(rows))]
train_ds = Dataset.from_list([to_example(r) for r in rows[:8000]])
eval_ds = Dataset.from_list([to_example(r) for r in rows[8000:8200]])   # 学習に使わない 200 件（損失の比較用）
heldout = rows[8200:8240]                                               # 学習に使わない 40 件（応答の比較用）
print(len(rows), "件のうち 学習", len(train_ds), "件")
print(train_ds[0])

15011 件 {'open_qa': 3742, 'general_qa': 2191, 'classification': 2136, 'closed_qa': 1773, 'brainstorming': 1766, 'information_extraction': 1506, 'summarization': 1188, 'creative_writing': 709}
12837 件のうち 学習 8000 件
{'prompt': [{'content': 'マルディグラで有名なアメリカの都市は？', 'role': 'user'}], 'completion': [{'content': 'ニューオーリンズはマルディグラで有名である。', 'role': 'assistant'}]}


In [4]:
# 自動評価の指示: 決まった形を守ったかを規則で判定できる指示を 6 種類 × 10 問用意する（IFEval のような考え方）
def n_bullets(t):
    return sum(bool(re.match(r"\s*([-・*•]|\d+[.)．、])", l)) for l in t.splitlines()) == 3
def strip_fence(t):
    return re.sub(r"^```(json)?\s*|\s*```$", "", t.strip())
def is_json(t, keys):
    try:
        d = json.loads(strip_fence(t))
        return isinstance(d, dict) and set(d) == set(keys)
    except ValueError:
        return False

CHECKS = {
    "箇条書き 3 つ": ("{}を 3 つ、箇条書きで挙げてください。説明は不要です。", n_bullets,
                   ["夏に食べたい果物", "朝の時間を有効に使う方法", "図書館でできること", "雨の日に家の中でできる遊び", "自転車に乗るときの注意点",
                    "旅行の持ち物", "部屋を片付けるこつ", "野菜を使った料理", "眠れないときに試すこと", "冬に楽しめるスポーツ"]),
    "30 文字以内": ("{}を 30 文字以内で説明してください。", lambda t: 0 < len(t) <= 30,
                 ["虹ができる仕組み", "冷蔵庫の役割", "俳句とは何か", "光合成", "地震が起きたときに最初にすること",
                  "月の満ち欠け", "辞書の使い道", "電子レンジの仕組み", "消しゴムで字が消える理由", "植物に水が必要な理由"]),
    "英語で": ("次の質問に英語で答えてください。{}", lambda t: len(re.findall(r"[A-Za-z]", t)) >= 3 and not re.search(r"[ぁ-んァ-ン一-龥]", t),
              ["日本でいちばん高い山は何ですか。", "1 年は何日ですか。", "好きな季節とその理由を教えてください。", "お茶とコーヒーの違いは何ですか。", "猫はどんな動物ですか。",
               "朝ごはんに何を食べるとよいですか。", "雨はなぜ降るのですか。", "図書館では何ができますか。", "健康のためにできることを教えてください。", "富士山はどこにありますか。"]),
    "はい/いいえ": ("次の質問に「はい」か「いいえ」の一言だけで答えてください。{}", lambda t: re.sub(r"[。.!！\s]", "", t) in ("はい", "いいえ"),
                 ["水は 100 度で沸騰しますか。", "太陽は西から昇りますか。", "1 週間は 7 日ですか。", "ペンギンは空を飛べますか。", "東京は日本の首都ですか。",
                  "氷は水に浮きますか。", "クジラは魚ですか。", "1 年は 12 か月ですか。", "月は自分で光っていますか。", "人間の心臓は 1 つですか。"]),
    "数字だけ": ("次の問いの答えを数字だけで答えてください。{}", lambda t: re.fullmatch(r"-?\d+(\.\d+)?", unicodedata.normalize("NFKC", t).strip(" 。.")) is not None,
               ["12 + 30 は？", "1 ダースは何個？", "1 時間は何分？", "7 × 8 は？", "三角形の内角の和は何度？",
                "1 キロメートルは何メートル？", "100 − 37 は？", "1 日は何時間？", "正方形の辺の数は？", "15 ÷ 3 は？"]),
    "JSON": ("次の文から情報を取り出し、キーが name と age の JSON だけを出力してください。{}", lambda t: is_json(t, ["name", "age"]),
             ["田中さんは 28 歳です。", "佐藤花子は今年 35 歳になった。", "12 歳の鈴木くんがサッカーをしている。", "山本さん（41 歳）は会社員です。", "45 歳の高橋さんが受付にいた。",
              "伊藤さんは 19 歳の大学生です。", "渡辺さん、52 歳。", "中村さんは先月 30 歳になりました。", "7 歳の小林さんは小学生だ。", "加藤さんの年齢は 63 歳です。"]),
}
if_items = [(kind, tmpl.format(x), check) for kind, (tmpl, check, xs) in CHECKS.items() for x in xs]
kinds = [k for k, _, _ in if_items]
if_prompts = [[{"role": "user", "content": p}] for _, p, _ in if_items]
dolly_prompts = [to_example(r)["prompt"] for r in heldout]

@torch.no_grad()
def generate(model, prompts, max_new_tokens=256, batch_size=40):
    """貪欲法で生成し、応答と「</s> を出して止まったか」を返す"""
    model.eval()
    tokenizer.padding_side = "left"
    outs, stopped = [], []
    for i in range(0, len(prompts), batch_size):
        enc = tokenizer.apply_chat_template(prompts[i:i + batch_size], add_generation_prompt=True, return_tensors="pt",
                                            return_dict=True, padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1,
                             pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        gen = out[:, enc["input_ids"].shape[1]:]
        outs += [tokenizer.decode(g, skip_special_tokens=True).strip() for g in gen]
        stopped += [(g == tokenizer.eos_token_id).any().item() for g in gen]
    return outs, np.array(stopped)

def evaluate(model, label):
    t = time.time()
    if_out, if_stop = generate(model, if_prompts)
    d_out, d_stop = generate(model, dolly_prompts)
    hits = np.array([check(o) for (_, _, check), o in zip(if_items, if_out)])
    print(f"{label}: 指示に従った割合 {hits.mean():.3f}、止まった割合 {np.r_[if_stop, d_stop].mean():.3f}（{time.time() - t:.0f} 秒）")
    return {"if_out": if_out, "hits": hits, "d_out": d_out, "stopped": np.r_[if_stop, d_stop]}

results = {"学習前": evaluate(model, "学習前（ベースモデル）")}   # Colab の T4 で 1〜2 分
for k in ["箇条書き 3 つ", "はい/いいえ", "JSON"]:
    i = kinds.index(k)
    print(f"\n[{if_items[i][1]}]\n→ {results['学習前']['if_out'][i][:200]}")

# 参考: チャットの形式を使わず、ふつうの文章の続きとして書かせる
enc = tokenizer("質問: 日本でいちばん高い山は何ですか。\n答え:", return_tensors="pt").to(model.device)
out = model.generate(**enc, max_new_tokens=60, do_sample=False, repetition_penalty=1.1, pad_token_id=tokenizer.pad_token_id)
print("\n[文章の続きとして]", repr(tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)))

学習前（ベースモデル）: 指示に従った割合 0.150、止まった割合 0.020（17 秒）

[夏に食べたい果物を 3 つ、箇条書きで挙げてください。説明は不要です。]
→ The following is the code for the problem:
```python
def find_max_profit(prices):
    """
    Find the maximum profit that can be made from buying and selling a stock given an array of daily prices.
 

[次の質問に「はい」か「いいえ」の一言だけで答えてください。水は 100 度で沸騰しますか。]
→ Answer

#include <bits/stdc++.h>
using namespace std;
const int N = 2e5 + 7, M = 3e6 + 7;
int a[N], b[M];
long long ans;
void solve() {
  int n, m;
  cin >> n >> m;
  for (int i = 1; i <= n; ++i) cin 

[次の文から情報を取り出し、キーが name と age の JSON だけを出力してください。田中さんは 28 歳です。]
→ ```python
def find_max_product(nums):
    """
    Given a list of integers, find the maximum product that can be obtained from any three integers in the list.
    The list may contain both positive an

[文章の続きとして] ' 日本でいちばん高い山は、富士山です。\n\n質問: 日本でいちばん低い山は何ですか。\n答え: 日本でいちばん低い山は、阿蘇山です。\n\n質問: 日本でいちばん大きい湖は何ですか。\n答え: 日本でいちばん大きい湖は、琵琶湖です。\n'


In [5]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])   # すべての線形層に低ランク行列を足す
args = SFTConfig(
    output_dir="sft-out", num_train_epochs=1, per_device_train_batch_size=8, gradient_accumulation_steps=2,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=10, max_length=512,
    completion_only_loss=True,    # prompt-completion 形式のデータでは既定でも True。応答 (completion) の部分だけに損失をかける
    packing=False,                # 複数の例を 1 本の系列に詰めない（下の注意書きを参照）
    logging_steps=50, save_strategy="no", report_to="none", seed=0,
    gradient_checkpointing=False, bf16=BF16, fp16=not BF16)   # SFTConfig は bf16=True と gradient_checkpointing=True が既定なので明示する
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds, processing_class=tokenizer, peft_config=lora)
trainer.model.print_trainable_parameters()

# 損失をかける部分の確認: SFTTrainer が作った labels で、プロンプトの部分は -100（損失を計算しない）になっている
ex = trainer.train_dataset[0]
ids, labels = np.array(ex["input_ids"]), np.array(ex["labels"])
print("損失なし:", repr(tokenizer.decode(ids[labels == -100])[-60:]))
print("損失あり:", repr(tokenizer.decode(ids[labels != -100])))
lengths = np.array([len(x) for x in trainer.train_dataset["input_ids"]])
print(f"トークン数: 平均 {lengths.mean():.0f}、最大 {lengths.max()}")
# 学習と同じくランダムに 8 件ずつまとめたときに、パディングが占める割合（packing をすると減らせる分）
pad = [1 - b.sum() / (len(b) * b.max()) for b in np.array_split(lengths[rng.permutation(len(lengths))], len(lengths) // 8)]
print(f"パディングの割合（ランダムに 8 件ずつ）: {np.mean(pad):.2f}")

trainable params: 10,076,160 || all params: 803,124,480 || trainable%: 1.2546
損失なし: '<|user|>マルディグラで有名なアメリカの都市は？</s><|assistant|>'
損失あり: 'ニューオーリンズはマルディグラで有名である。</s>'
トークン数: 平均 99、最大 409
パディングの割合（ランダムに 8 件ずつ）: 0.54


In [6]:
from trl.trainer.sft_trainer import DataCollatorForLanguageModeling

# packing の確認: trl の packing は複数の例を 1 本につなぎ、位置 (position_ids) を例ごとに 0 から振り直す。
# 2 件をつないだときの 2 件目の出力が、2 件目だけを入れたときと同じになるか（前の例を見ていないか）を確かめる
a, b = trainer.train_dataset[0]["input_ids"], trainer.train_dataset[1]["input_ids"]
batch = DataCollatorForLanguageModeling(pad_token_id=tokenizer.pad_token_id, padding_free=True)([{"input_ids": a}, {"input_ids": b}])
ids, pos = batch["input_ids"].to(model.device), batch["position_ids"].to(model.device)
trainer.model.eval()
with torch.no_grad():
    alone = trainer.model(input_ids=torch.tensor([b], device=model.device)).logits[0].float()
    for label, kw in [("位置を振り直す（学習時と同じくキャッシュなし）", dict(position_ids=pos, use_cache=False)),
                      ("単純につなぐだけ", dict(use_cache=False))]:
        packed = trainer.model(input_ids=ids, **kw).logits[0, len(a):].float()
        print(f"2 件目の logits の最大の差（{label}）: {(packed - alone).abs().max().item():.2f}")
print("アテンションの実装:", model.config._attn_implementation)

2 件目の logits の最大の差（位置を振り直す（学習時と同じくキャッシュなし））: 0.04
2 件目の logits の最大の差（単純につなぐだけ）: 16.02
アテンションの実装: sdpa


In [7]:
loss_before = trainer.evaluate()["eval_loss"]   # 学習前（LoRA の B は 0 で初期化されるので、ベースモデルと同じ）
t = time.time()
trainer.train()   # Colab の T4 で 10〜15 分
for log in trainer.state.log_history:
    if "loss" in log:
        print(f"step {log['step']:3d}  loss {log['loss']:.3f}")
print(f"評価データ 200 件の応答部分の損失: 学習前 {loss_before:.3f} → 学習後 {trainer.evaluate()['eval_loss']:.3f}")
print(f"学習 {time.time() - t:.0f} 秒、GPU メモリの最大使用量 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
# 参考: 全パラメータを Adam で学習すると、重み・勾配・Adam の状態だけで 1 パラメータあたり 16 バイト程度（混合精度の場合）
n_all = sum(p.numel() for p in trainer.model.parameters())
n_train = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
print(f"学習するパラメータ {n_train / 1e6:.1f}M（全体の {n_train / n_all:.2%}）。全パラメータを学習する場合の目安 {n_all * 16 / 1e9:.1f} GB + 活性化")

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,2.945343,0,2.766362,0.462437,0.000000


Step,Training Loss
50,2.478083
100,2.337267
150,2.323227
200,2.266523
250,2.288683
300,2.395528
350,2.361322
400,2.263231
450,2.325952
500,2.260117


step  50  loss 2.478
step 100  loss 2.337
step 150  loss 2.323
step 200  loss 2.267
step 250  loss 2.289
step 300  loss 2.396
step 350  loss 2.361
step 400  loss 2.263
step 450  loss 2.326
step 500  loss 2.260


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
2.260117,2.385500,500,2.416774,0.523522,790354.000000


評価データ 200 件の応答部分の損失: 学習前 2.945 → 学習後 2.386
学習 209 秒、GPU メモリの最大使用量 8.6 GB
学習するパラメータ 10.1M（全体の 1.25%）。全パラメータを学習する場合の目安 12.8 GB + 活性化


In [8]:
# 学習後のモデルと公式の指示チューニング版を、同じ指示で評価する
model = trainer.model
results["学習後 (SFT)"] = evaluate(model, "学習後（LoRA で SFT）")   # 公式版の読み込みと合わせて Colab の T4 で 1〜3 分
inst = AutoModelForCausalLM.from_pretrained(inst_name, dtype=DTYPE, device_map="auto")
results["公式の instruct"] = evaluate(inst, "公式の指示チューニング版")

def rate_ci(hits):   # 割合と、ブートストラップによる 95% 信頼区間
    boot = hits[np.random.default_rng(0).integers(0, len(hits), (2000, len(hits)))].mean(1)
    return f"{hits.mean():.2f} [{np.percentile(boot, 2.5):.2f}, {np.percentile(boot, 97.5):.2f}]"

table = pd.DataFrame({label: {**{k: r["hits"][[i for i, x in enumerate(kinds) if x == k]].mean() for k in CHECKS},
                              "指示に従った割合（60 問）": rate_ci(r["hits"]), "止まった割合（100 問）": r["stopped"].mean().round(2),
                              "応答の平均の長さ（文字）": round(np.mean([len(o) for o in r["if_out"] + r["d_out"]]))}
                      for label, r in results.items()})
print(table.to_string())
print(f"GPU メモリの最大使用量 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

学習後（LoRA で SFT）: 指示に従った割合 0.200、止まった割合 0.920（14 秒）
公式の指示チューニング版: 指示に従った割合 0.200、止まった割合 0.930（11 秒）
                              学習前          学習後 (SFT)       公式の instruct
箇条書き 3 つ                      0.0                0.9                0.2
30 文字以内                       0.0                0.2                0.0
英語で                           0.9                0.0                0.0
はい/いいえ                        0.0                0.0                0.0
数字だけ                          0.0                0.1                0.0
JSON                          0.0                0.0                1.0
指示に従った割合（60 問）  0.15 [0.07, 0.25]  0.20 [0.12, 0.30]  0.20 [0.10, 0.32]
止まった割合（100 問）                0.02               0.92               0.93
応答の平均の長さ（文字）                  826                 92                251
GPU メモリの最大使用量 8.6 GB


In [9]:
# 同じ指示への 3 つの応答を並べる（形を指定した指示 2 問と、学習に使っていない dolly の指示 2 問）
for p, outs in [(if_items[i][1], [r["if_out"][i] for r in results.values()]) for i in [kinds.index("30 文字以内"), kinds.index("数字だけ")]] + \
               [(dolly_prompts[i][0]["content"], [r["d_out"][i] for r in results.values()]) for i in [0, 1]]:
    print("■", p[:150])
    for label, o in zip(results, outs):
        print(f"  [{label}] {o[:150]}")
    print()

model.save_pretrained("sft-lora")   # 保存されるのは追加した LoRA の重みだけ
size = sum(os.path.getsize(f"sft-lora/{f}") for f in os.listdir("sft-lora") if f.endswith(".safetensors"))
print(f"LoRA の重みのファイル: {size / 1e6:.0f} MB")

■ 虹ができる仕組みを 30 文字以内で説明してください。
  [学習前] The answer is:

To understand the process of how a rainbow forms, we need to first know about light. Light is a form of energy that travels through sp
  [学習後 (SFT)] 虹は、太陽光が大気中の水滴に反射してできる。太陽光の波長は、空気中では赤から紫までだが、水滴の中に入ると、その波長は7色に分かれる。この7色は、太陽光の波長によって、それぞれ異なる色で見える。
  [公式の instruct] 虹は太陽光が水滴の中で屈折し、反射することで人間の目に届く現象です。まず、太陽からの光が大気中の水滴に接触すると屈折し、一部は真上に、残りは下に曲がります。この屈折した光が水滴内で反射を繰り返すと、赤、橙、黄、緑、青、藍、紫の7色の虹として現れます。それぞれの波長が水滴で異なる角度で反射するため、私

■ 次の問いの答えを数字だけで答えてください。12 + 30 は？
  [学習前] Let's solve the multi-choice question step by step.

$12+30=42$

Hence, the answer is (c) 42.

My Personal Notes arrow_drop_up Check out this Author's
  [学習後 (SFT)] 答えは32である。
  [公式の instruct] 答えは 42 です。

■ この文章で名前が挙がっている人たちは誰ですか？

ラウリ・ヨハネス・シルヴァーン（1932年3月13日生まれ）は、フィンランドのアイスホッケー選手。1952年冬季オリンピックの男子トーナメントに出場。
  [学習前] The question is: What was the name of the first person to be killed in a car crash?<|assistant|>John Doe<|user|>The question is: Who won the 1984 Worl
  [学習後 (SFT)] ラウリ・ヨハネス・

- 指示チューニング (SFT: supervised fine-tuning) は、指示と応答の組で、応答の部分を次のトークンの予測として学習させること。上の出力では、ベースモデルはチャットの形式で渡した指示に英語の文章やプログラムを書き続け、`</s>` を出して止まったのは 100 問中 2 問だった。ふつうの文章の続きとしてなら答えるが、次の質問と答えを自分で作り続ける。8,000 件・1 エポックの SFT のあとは止まった割合が 0.92 になり（公式の指示チューニング版は 0.93）、学習に使っていない 200 件の応答部分の損失は 2.945 → 2.386 に下がった。なお、ベースモデルのモデルカードには、事前学習の最後の段階で「さまざまな応用タスクの性能を上げるための少量のデータ」で学習したとあり、純粋に Web のテキストだけで学習したモデルではない。
- 決まった形を守ったかの自動評価（60 問）では、指示に従った割合は学習前 0.15、学習後 0.20、公式版 0.20 で、95% 信頼区間が重なり、この問題数では差があるとは言えない。学習前の 0.15 は、何を聞かれても英語を書くので「英語で」の 10 問中 9 問が当たっただけである。学習後は「箇条書き 3 つ」が 0.9 と公式版 (0.2) より高い一方、「JSON」は 0（公式版は 1.0）で、「はい/いいえ」の一言は 3 つとも 0 だった。SFT で身につくのは主に学習データの形と文体で、学習後の応答は短く（平均 92 文字、公式版は 251 文字）、学習データと同じである調になった。dolly には JSON で答える例がほとんどない。「12 + 30」に「答えは32である。」と答えたり、参考文を答えの代わりにそのまま写したりするように、0.5B のモデルに 8,000 件学習させただけでは中身の正しさは上がらない。決まった形で答えさせたいなら、その形の例を学習データに入れる。中身の良し悪しを比べるには、LLM の評価と分析の章の「手元の LLM に生成文を採点させる (LLM-as-a-judge)」のような判定を使う。
- 損失をかける部分: 会話形式の prompt-completion（`prompt` と `completion` にメッセージのリスト）のデータを渡すと、`SFTTrainer` はプロンプトの部分の `labels` を -100 にして、応答とその後の `</s>` だけに損失をかける（`completion_only_loss` は、この形式では既定で True）。`</s>` を学習するので、答えたあとに止まれるようになる。複数ターンの会話（`messages` の列）で応答だけに損失をかける `assistant_only_loss=True` は、チャットテンプレートに応答の範囲を示す `{% generation %}` のタグが要り、sarashina2.2 の公式のテンプレートにも、ここで設定したテンプレートにもない（trl は、学習用のテンプレートを用意してある一部のモデルでは自動で差し替えるが、sarashina2.2 はその対象ではない）。ベースモデルにはチャットテンプレートがないので、公式版と同じ形式のものを設定した（`<|user|>` などはベースモデルの語彙にも同じ ID で追加されている。特殊トークンの扱いではないので、`skip_special_tokens=True` でも消えず、上の学習前の出力にもそのまま出ている）。こうすると、学習したモデルを公式版と同じ使い方で動かせる。
- packing: 長さがばらばらな例を 8 件ずつまとめると、上の出力ではトークンの 54% がパディングになる。`packing=True` にすると複数の例を 1 本の系列に詰めてパディングを減らせる。trl は例ごとに位置を 0 から振り直し、transformers 5 は位置の切れ目から例どうしのアテンションを遮る（学習時のようにキャッシュを使わない場合）。上の確認では、位置を振り直したときの 2 件目の logits の差は 0.04（float16 の誤差の程度）、単純につなぐと 16.02 で、前の例が見えてしまう。trl の説明では packing（padding-free）は FlashAttention 2/3 が前提とされているが、この版の transformers では sdpa でも上のとおり遮られた（版によって変わりうる）。ここでは、例ごとに扱えて分かりやすいように packing を使っていない。packing をすると 1 本に複数の例が入り、同じデータでもステップ数が減るので、学習率やステップ数を合わせ直す必要がある（ここでは時間を比べていない）。
- LoRA (Hu et al., 2021) は、元の重みを固定し、線形層に低ランクの行列の積 BA（ランク `r`）を足して、A と B だけを学習する。B は 0 で初期化するので、学習前は元のモデルと同じ出力になる。`lora_alpha / r` が足す量の倍率。`target_modules` はアテンションの 4 つの射影と MLP の 3 つの線形層で、学習するパラメータは 10.1M（全体の 1.25%）、保存される重みは 40MB だった。学習中の GPU メモリの最大使用量は 8.6GB。全パラメータを Adam で学習すると、混合精度では重み・勾配・Adam の状態だけで 1 パラメータあたり 16 バイト程度 (Rajbhandari et al., 2020)、このモデルで 12.8GB に活性化の分が加わり、T4 (16GB) では厳しい。4bit 量子化と組み合わせる (QLoRA) と、もっと大きなモデルも T4 で学習できる（`prepare_model_for_kbit_training` を使う）。
- 読み込みは `PeftModel.from_pretrained(ベースモデル, "sft-lora")`、推論を速くするなら `merge_and_unload()` で重みを統合する。LoRA の学習率は全パラメータの学習より高め（ここでは 2e-4）。`SFTConfig` は trl 1.18.0 では `bf16=True`、`gradient_checkpointing=True`、学習率 2e-5 が既定なので、T4 向けに fp16 を明示している。検証に使った GPU（T4 と同じ float16 の経路）では学習が 209 秒、全体で約 5 分だった。T4 では学習が 10〜15 分程度、全体で 20 分程度と見込んでいる（T4 では測っていない）。
- ベース → SFT → 選好や報酬による調整、という順番がふつうである。この章の「選好データで LLM を調整する (DPO)」と「正解を判定できる課題で LLM を強化学習する (GRPO)」は、公式の指示チューニング版から始めている（その学習方法はモデルカードに書かれていない）。ここで作ったモデルから始めるなら、`merge_and_unload()` で統合したモデルを渡し、トークナイザーはここで設定したチャットテンプレートを持つもの（`tokenizer.save_pretrained` で保存したもの）か公式版のものを使う。ベースモデルのトークナイザーのままではチャットテンプレートがない。SFT は「手本の応答をまねる」学習で、手本より良い応答は学べない。DPO は 2 つの応答のどちらが良いか、GRPO は正解かどうかで、手本のない改善を目指す。
- データ: llm-jp/databricks-dolly-15k-ja (CC BY-SA 3.0) は、Databricks の社員が書いた databricks-dolly-15k を LLM-jp が DeepL で訳したもので、訳の崩れも混じる。元のデータセットカードによると、要約や閉じた QA などの参考文 (context) は Wikipedia から引用したもので、引用の番号（[42] など）が残っていることがある。CC BY-SA は改変したデータを同じライセンスで公開することを求めるので、データを加工して配布するときは注意する。モデルはどちらも MIT License。トークナイザーの修正 (`load_tokenizer`) については、大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照。